In [ ]:
import numpy as np
import pandas as pd
import dai


def main(datasources, start_date, end_date):
    bar1m = datasources["bar1m"]

    sql = f"""
    WITH minute_base AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            date::TIME AS bar_time,
            open,
            low,
            close,
            bid_volume1,
            bid_volume2,
            bid_volume3,
            bid_volume4,
            bid_volume5,
            ask_volume1,
            ask_volume2,
            ask_volume3,
            ask_volume4,
            ask_volume5
        FROM {bar1m}
    ),
    minute_feature AS (
        SELECT
            date,
            instrument,
            bar_time,
            open,
            low,
            close,
            (
                bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                - ask_volume1 - ask_volume2 - ask_volume3 - ask_volume4 - ask_volume5
            )
            / NULLIF(
                bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5,
                0
            ) AS imbalance_1_5
        FROM minute_base
    ),
    daily_raw AS (
        SELECT
            date,
            instrument,

            arg_min(open, bar_time) AS open_first,
            arg_max(close, bar_time) AS close_last,
            min(low) AS low_min,

            avg(
                CASE
                    WHEN bar_time <= '09:45:00'::TIME
                    THEN close
                    ELSE NULL
                END
            ) AS morning_close_avg,

            avg(
                CASE
                    WHEN bar_time <= '09:45:00'::TIME
                    THEN imbalance_1_5
                    ELSE NULL
                END
            ) AS morning_imbalance,

            avg(
                CASE
                    WHEN bar_time >= '14:30:00'::TIME
                    THEN imbalance_1_5
                    ELSE NULL
                END
            ) AS tail_imbalance

        FROM minute_feature
        GROUP BY date, instrument
    ),
    daily_factor AS (
        SELECT
            date,
            instrument,

            (
                -1.0 * (morning_close_avg / NULLIF(open_first, 0) - 1.0)
                + (close_last / NULLIF(low_min, 0) - 1.0)
            ) AS path_raw,

            (
                morning_imbalance - tail_imbalance
            ) AS ob_raw

        FROM daily_raw
        WHERE open_first > 0
          AND close_last > 0
          AND low_min > 0
          AND morning_close_avg > 0
          AND morning_imbalance IS NOT NULL
          AND tail_imbalance IS NOT NULL
    )
    SELECT
        date,
        instrument,
        path_raw,
        ob_raw
    FROM daily_factor
    ORDER BY date, instrument
    """

    result = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    result = result[["date", "instrument", "path_raw", "ob_raw"]].copy()

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["instrument"] = result["instrument"].astype(str)
    result["path_raw"] = pd.to_numeric(result["path_raw"], errors="coerce")
    result["ob_raw"] = pd.to_numeric(result["ob_raw"], errors="coerce")

    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["date", "instrument", "path_raw", "ob_raw"])

    result["path_rank"] = result.groupby("date")["path_raw"].rank(pct=True)
    result["ob_rank"] = result.groupby("date")["ob_raw"].rank(pct=True)

    result["factor"] = 0.6 * result["path_rank"] + 0.4 * result["ob_rank"]

    result = result[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["date", "instrument", "factor"])
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)

    return result[["date", "instrument", "factor"]]